In [2]:
# 导入环境变量
from dotenv import load_dotenv
import os

load_dotenv(override=True)
BASE_URL = os.getenv("MINIMAX_BASE_URL")
API_KEY = os.getenv("MINIMAX_API_KEY")
MODEL_ID = os.getenv("MINIMAX_MODEL_ID")

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="anthropic",  # Mini-Max模型通过anthropic的模型规范进行交互
    model=MODEL_ID,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# 多轮对话的上下文记忆

In [6]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

print("--- 第一轮的对话 ---")
messages_round_1 = [
    SystemMessage(content="你是有个幽默的人，每次回复都要用：啊哈！开头"),
    HumanMessage(content="你好呀")
]

print(f"系统提示词：{messages_round_1[0].text}")
print(f"用户：{messages_round_1[1].text}")
response_1 = model.invoke(messages_round_1)
print(f"AI回复：{response_1.text}")

print("--- 第二轮的对话 ---")
messages_round_2 = messages_round_1 + [response_1, HumanMessage(content="讲个笑话"),]
print(f"用户：{messages_round_2[3].text}")
response_2 = model.invoke(messages_round_2)
print(f"AI回复：{response_2.text}")


--- 第一轮的对话 ---
系统提示词：你是有个幽默的人，每次回复都要用：啊哈！开头
用户：你好呀
AI回复：啊哈！你好呀！很高兴见到你！今天有什么好玩的话题，还是单纯来打个招呼？😄
--- 第二轮的对话 ---
用户：讲个笑话
AI回复：啊哈！程序员去相亲。

女生问：“你有房吗？”  
程序员：“有，GitHub。”

女生问：“你有车吗？”  
程序员：“有，Docker。”

女生问：“你有存款吗？”  
程序员：“有，都放在云端。”

女生沉默一会儿：“那你什么时候结婚？”  
程序员：“等我部署到生产环境。”

女生：“……”

程序员：“怎么了？”  
女生：“你先去相亲吧！” 😄


# 工具调用示例

In [ ]:
# 如果模型要调用工具，会输出类似下面的消息
model_tool_call = AIMessage(
    content="",
    tool_calls=[
        {"id": "call_123", "name": "calculator", "args": {"a": 2, "b": 3}},
    ]
)

# 外层 Harness 执行完工具会将结果写回
calculator_result = "5"  # 模拟工具执行结果
tool_feedback = ToolMessage(
    content=calculator_result,
    tool_call_id=model_tool_call.tool_calls[0]["id"],  # 与模型输出的调用工具id一致
)